In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
except:
    from google.colab import userdata
    wandb_api_key = userdata.get('WANDB_API_KEY')


In [ ]:
# Write the .netrc file
netrc_content = f"""machine api.wandb.ai
  login user
  password {wandb_api_key}
"""

with open("/root/.netrc", "w") as f:
    f.write(netrc_content)

# Make sure permissions are correct
import os
os.chmod("/root/.netrc", 0o600)

In [ ]:
%cd /content
!git clone https://github.com/BouncyButton/hippopotamus
!pip install wget
!mkdir datasets
%cd datasets

In [ ]:
!python /content/hippopotamus/datasets/Dataset102_MNI/create_mni_dataset.py

# UNETR++

This looks tricky to run because it needs a precise running env. Cannot run easily on the cloud.

In [ ]:
%cd /content/hippopotamus/baselines/unetr_plus_plus

In [ ]:
!apt-get install python3.8-venv -y
!python3.8 -m venv py38


In [ ]:
!py38/bin/python --version

In [ ]:
!py38/bin/pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113


In [ ]:
!py38/bin/pip install -r requirements.txt

In [ ]:
!pwd

In [ ]:
!mkdir /content/preprocessed
!mkdir /content/preprocessed/Task102_MNI

In [ ]:
!mkdir /content/output_folder

In [ ]:
%cd /content/

In [ ]:
%env RESULTS_FOLDER=/content/output_folder
%env unetr_pp_preprocessed=/content/datasets/preprocessed/
%env unetr_pp_raw_data_base=/content/datasets/


In [ ]:
!mkdir /content/datasets/unetr_pp_raw_data

In [ ]:
!mv /content/datasets/Dataset102_MNI /content/datasets/unetr_pp_raw_data/Dataset102_MNI

In [ ]:
!mv /content/datasets/unetr_pp_raw_data/Dataset102_MNI /content/datasets/unetr_pp_raw_data/Task102_MNI

In [ ]:
json_content_v2 = """{
  "channel_names": {
    "0": "MRI"
  },
  "labels": {
    "background": 0,
    "anterior": 1,
    "posterior": 2
  },
  "numTraining": 260,
  "file_ending": ".nii.gz"
}"""

In [ ]:
%cd /content/hippopotamus/baselines/unetr_plus_plus/

In [ ]:
import os
import json

# Change directory to /content/ first
os.chdir('/content/')

base_path = '/content/datasets/unetr_pp_raw_data/Task102_MNI'
images_tr_dir = os.path.join(base_path, 'imagesTr')
labels_tr_dir = os.path.join(base_path, 'labelsTr')
images_ts_dir = os.path.join(base_path, 'imagesTs')
dataset_json_path = os.path.join(base_path, 'dataset.json')

# Ensure the base path exists
os.makedirs(base_path, exist_ok=True)

# Get file lists
image_files_tr_full = [f for f in os.listdir(images_tr_dir) if f.endswith(".nii.gz") and not f.startswith("._")]
label_files_tr_full = [f for f in os.listdir(labels_tr_dir) if f.endswith(".nii.gz") and not f.startswith("._")]
test_image_files_full = [f for f in os.listdir(images_ts_dir) if f.endswith(".nii.gz") and not f.startswith("._")] if os.path.exists(images_ts_dir) else []

print(f"Number of image files in {images_tr_dir}: {len(image_files_tr_full)}")
print(f"Number of label files in {labels_tr_dir}: {len(label_files_tr_full)}")
print(f"Number of test image files in {images_ts_dir}: {len(test_image_files_full)}")

# Print first few to verify naming convention
print(f"\nFirst 5 image files in {images_tr_dir}: {sorted(image_files_tr_full)[:5]}")
print(f"First 5 label files in {labels_tr_dir}: {sorted(label_files_tr_full)[:5]}")
print(f"First 5 test image files in {images_ts_dir}: {sorted(test_image_files_full)[:5]}")


training_cases = []
for image_file in sorted(image_files_tr_full):
    # Assuming image files are like MNI_XXX_0000.nii.gz
    # The base name is without the '_0000' and '.nii.gz'
    image_base_name = image_file.replace('_0000.nii.gz', '').replace('.nii.gz', '')

    label_filename = None

    # Check for label file with '_0000' suffix (as implied by previous nnUNet error for MSD)
    potential_label_with_suffix = image_base_name + '_0000.nii.gz'
    if os.path.exists(os.path.join(labels_tr_dir, potential_label_with_suffix)):
        label_filename = potential_label_with_suffix
    else:
        # Check for label file without '_0000' suffix
        potential_label_without_suffix = image_base_name + '.nii.gz'
        if os.path.exists(os.path.join(labels_tr_dir, potential_label_without_suffix)):
            label_filename = potential_label_without_suffix
        else:
            print(f"Warning: No matching label found for image {image_file} in {labels_tr_dir}. Tried {potential_label_with_suffix} and {potential_label_without_suffix}. Skipping this case.")
            continue # Skip this training case if no label is found

    training_cases.append({
        "image": f"./imagesTr/{image_file.replace("_0000", '')}",
        "label": f"./labelsTr/{label_filename}"
    })

test_cases = []
for test_file in sorted(test_image_files_full):
    test_cases.append(f"./imagesTs/{test_file}")

dataset_name = "MNI"
num_training = len(training_cases)
num_test = len(test_cases)

# Define dataset.json content based on nnUNet format and context
dataset_json_content = {
    "labels": {
        "0": "background",
        "1": "CA1-3",
        "2": "Subiculum",
        "3": "CA4-DG"
    },
    "channel_names": {
        "0": "MRI"
    },
    "name": dataset_name,
    "numTraining": num_training,
    "numTest": num_test,
    "file_ending": ".nii.gz",
    "description": "Dynamically generated dataset.json for Task102_MNI for hippocampus segmentation.",
    "licence": "see challenge website",
    "modality": {
        "0": "MRI"
    },
    "reference": "see challenge website",
    "release": "0.0",
    "tensorImageSize": "4D",
    "training": training_cases,
    "test": test_cases
}

# Write the JSON content to the file
with open(dataset_json_path, 'w') as f:
    json.dump(dataset_json_content, f, indent=4)

print(f"\nGenerated dataset.json for Task102_MNI at: {dataset_json_path}")
print(f"Number of training cases included: {num_training}")
print(f"Number of test cases included: {num_test}")


In [ ]:
%cd /content/hippopotamus/baselines/unetr_plus_plus/

In [ ]:
!py38/bin/python -m unetr_pp.experiment_planning.nnFormer_plan_and_preprocess -t 102

In [ ]:
!py38/bin/pip install --force-reinstall matplotlib

In [ ]:
%cd /content/hippopotamus/baselines/unetr_plus_plus

In [ ]:
!git pull

In [ ]:
!timeout 15 py38/bin/python -m unetr_pp.run.run_training 3d_fullres unetr_pp_trainer_general_purpose 102 0 --crop_size 48 56 40

In [ ]:
!py38/bin/python /content/hippopotamus/datasets/fix_unetrpp_splits_pkl_to_json.py -i /content/datasets/preprocessed/Task102_MNI/splits_final.pkl -o /content/datasets/preprocessed/Task102_MNI/splits_final.json

In [ ]:
!python /content/hippopotamus/datasets/fix_unetrpp_splits_json.py -i /content/datasets/preprocessed/Task102_MNI/splits_final.json -o /content/datasets/preprocessed/Task102_MNI/splits_final_fixed.json --dataset MNI

In [ ]:
!py38/bin/python /content/hippopotamus/datasets/fix_unetrpp_splits_json_to_pkl.py -i /content/datasets/preprocessed/Task102_MNI/splits_final_fixed.json -o /content/datasets/preprocessed/Task102_MNI/splits_final.pkl

In [ ]:
!py38/bin/python -m unetr_pp.run.run_training 3d_fullres unetr_pp_trainer_general_purpose 102 4 --crop_size 48 56 40
!python /content/hippopotamus/baselines/save_unetrpp_run.py --model-path /content/output_folder/unetr_pp/3d_fullres/Task102_MNI/unetr_pp_trainer_general_purpose__unetr_pp_Plansv2.1/fold_4 --fold 4 --dataset MNI



In [ ]:
!py38/bin/python -m unetr_pp.run.run_training 3d_fullres unetr_pp_trainer_general_purpose 102 3 --crop_size 48 56 40
!python /content/hippopotamus/baselines/save_unetrpp_run.py --model-path /content/output_folder/unetr_pp/3d_fullres/Task102_MNI/unetr_pp_trainer_general_purpose__unetr_pp_Plansv2.1/fold_3 --fold 3 --dataset MNI

In [ ]:
!py38/bin/python -m unetr_pp.run.run_training 3d_fullres unetr_pp_trainer_general_purpose 102 2 --crop_size 48 56 40
!python /content/hippopotamus/baselines/save_unetrpp_run.py --model-path /content/output_folder/unetr_pp/3d_fullres/Task102_MNI/unetr_pp_trainer_general_purpose__unetr_pp_Plansv2.1/fold_2 --fold 2 --dataset MNI

In [ ]:
!py38/bin/python -m unetr_pp.run.run_training 3d_fullres unetr_pp_trainer_general_purpose 102 1 --crop_size 48 56 40
!python /content/hippopotamus/baselines/save_unetrpp_run.py --model-path /content/output_folder/unetr_pp/3d_fullres/Task102_MNI/unetr_pp_trainer_general_purpose__unetr_pp_Plansv2.1/fold_1 --fold 1 --dataset MNI